<img src="../Images/DSC_Logo.png" style="width: 400px;">

This notebook applies a workflow that is built on [faster-whisper](https://github.com/SYSTRAN/faster-whisper) and [pyannote.audio](https://github.com/pyannote/pyannote-audio), with installation procedures based on their official GitHub repositories (accessed September 25, 2025). This is one common Whisper-based setup among several: For example, [Whisper](https://github.com/openai/whisper) can also be run directly (without faster-whisper) or via more integrated pipelines such as [WhisperX](https://github.com/m-bain/whisperX). 

The overall approach that is presented in this notebook consists of: 
1. **Automatic Speech Recognition (ASR)** with Whisper (speech-to-text transcription)
2. **Speaker diarization** with pyannote (who speaks when)
3. **Time-based merging** (assign speaker labels to the transcript)

This is similar to what runs in the background of tools like [noScribe](https://github.com/kaixxx/noScribe). 

>If you are **only interested in a plain transcript (no speaker labels)**, you can use the separate **`faster-whisper` Jupyter notebook** instead. There, pyannote, the Hugging Face setup, and the merging steps are skipped entirely.

>**Using this notebook:** The workflow can be adapted throughout, but to test ASR and speaker diarization with the default settings, only the parameters marked with **`!`** need to be changed or checked.


# 1. One-Time Setup: Install Software & Hugging Face Account

## 1.1 Install Software

The code below installs the **Python packages** listed in "requirements.txt" into the Python environment your Jupyter notebook is using. It ensures all needed libraries (and versions) are available so the notebook can run without import errors.

> `!` Check that you once installed the required packages in your working environment.

In [ ]:
#%pip install -r ../requirements.txt

## 1.2 Hugging Face Account

Pyannote diarization requires a Hugging Face account and model access. The setup consists of three steps:
1. Create a **Hugging Face account** (if you don’t have one yet): [Hugging Face Account](https://huggingface.co/)
2. Create an **access token** (needed to download and run the diarization model):
Click your profile icon → Settings → Access Tokens → create a new token with read permission (no write permission needed; easiest: select token type "Read").

>Important: Save your token somewhere so you cann look it up later, but don't share your token.

3. **Request access** to the following model repositories, review the conditions, enter your information, and click "Agree and access repository":
- [speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1)
- [segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0)
- [speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1)

>If you encounter any error messages when applying speaker diarization to this notebook, please check the latest instructions in the [pyannote.audio](https://github.com/pyannote/pyannote-audio) repository.

> `!` Required setup and you need to replace `# "ENTER_YOUR_TOKEN"` with your token

In [ ]:
own_token = # "ENTER_YOUR_TOKEN"

# 2. Import Packages

In [ ]:
# Optional to avoid warnings:
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)

In [ ]:
import os                                            # work with file/folder paths
from datetime import timedelta                       # format/handle time durations
import subprocess                                    # run external commands from Python (here: call ffmpeg)
import imageio_ffmpeg                                # provides an ffmpeg executable we can call from Python (audio conversion)
import torch                                         # PyTorch backend (used by pyannote; also lets us check GPU)
from faster_whisper import WhisperModel              # speech-to-text (ASR)
from faster_whisper import BatchedInferencePipeline  # optional: faster transcription on GPU (batching)
import torchaudio                                    # load audio into memory: waveform (tensor) + sample rate
from pyannote.audio import Pipeline                  # speaker diarization pipeline (“who speaks when”)

# path to the ffmpeg executable used by subprocess:
ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()  

You can check the installed PyTorch version and whether your environment has access to a GPU.

Meaning of the output:
- CUDA available: False = No GPU access
- CUDA available: True = GPU access

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# 3. Setup

## 3.1 Runtime Settings

Automatically set device and compute type depending on **hardware availability**. You don't need to know whether your hardware has a CPU or a GPU. This is **checked and selected automatically** here. 

The compute type tells the engine what kind of "number format" it should use internally while running the model. Different formats trade off speed and resource use. Batch size controls how many audio chunks are processed at once. On a GPU, a larger batch size can speed things up. On a CPU, batch size is usually kept at 1 because larger values typically don’t help.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    compute_type = "float16"  # GPU: usually fastest
    batch_size = 16           # GPU: process several chunks at once (reduce if you get errors)
else:
    compute_type = "int8"     # CPU: usually fastest
    batch_size = 1            # CPU: process one chunk at a time

## 3.2 Select Audio File

With Python, you can easily transcribe **multiple files by looping over a list of paths** (e.g., all files in a folder) and applying the same steps to each file. In this notebook, we keep things simple and specify a single audio file. 

Below, we provide the **relative path to one audio file**. Both .wav and .mp3 files work because the transcription library uses ffmpeg under the hood to read many common audio formats. In addition, in the next step we explicitly convert the audio to a standardized format to ensure consistent processing throughout the notebook. 

To switch between example files, uncomment exactly one pair (`file_name` & `audio_file`) and keep all others commented out. To use your own audio, add the file to `../Data_Raw/`, then set `file_name` & `audio_file` in the code below to the file’s name and relative path (and comment out the other examples).

> `!` Check the selected file and optionally select a different one (comment out while uncommenting the other one)

In [ ]:
#file_name = "File-A"
#audio_file = "../Data_Raw/File-A_buffy/shortened_Buffy_Seas01-Epis01.en.wav"

#file_name = "File-B"
#audio_file = "../Data_Raw/File-B_moon-landing/shortened_CA138.mp3"

file_name = "File-C"
audio_file = "../Data_Raw/File-C_qualitative-interview-en/shortened_JG-20170508_Interview-recording.wav"

#file_name = "File-D"
#audio_file = "../Data_Raw/File-D_Bremen-guide-low-saxon/shortened_audioguide-2025-platt-01.wav"

#file_name = "File-E"
#audio_file = "../Data_Raw/File-E_common-voice/common_voice_pcm_41315452.mp3"   # Note: multiple files available

# 4. Preprocess Audio File

Whisper and pyannote can read many audio formats and handle basic **resampling internally**. "Basic resampling" here means that Whisper and pyannote can automatically adjust the sample rate of your audio file to match the model's required rate, so you do not need to manually convert it beforehand. 

In this notebook, we still apply one light **preprocessing** step: we standardize the audio to a 16 kHz mono WAV.

More **advanced preprocessing** (denoising, volume normalization, echo removal, speech separation) is usually optional. It might be beneficial if you notice clear problems, such as strong background noise/echo, very uneven volume, very long silences, or heavy overlapping speech.

## 4.1 Run ffmpeg

The audio file is converted once to a 16 kHz mono WAV. This **standardized audio file** is then used for both Whisper and pyannote so they share the exact same audio time base. This ensures that Whisper and pyannote use the exact same audio file and time base, which makes the later alignment/merging step more reliable (see Sect. 8).

In [ ]:
audio_16k = "../Data_Preprocessed/audio_16k_mono.wav"

# Convert with ffmpeg (standardize audio for consistent processing):
# -y                 -> overwrite output file if it already exists
# -hide_banner       -> hide ffmpeg version banner
# -loglevel error    -> show only errors (no progress/info output)
# -i <input>         -> input audio file (e.g., .mp3 or .wav)
# -ac 1              -> convert to mono (1 audio channel)
# -ar 16000          -> resample to 16,000 Hz (common format for speech models)

subprocess.run(
    [ffmpeg, "-y",
     "-hide_banner",
     "-loglevel", "error",
     "-i", audio_file,
     "-ac", "1", "-ar", "16000",
     audio_16k],
    check=True
)

print("Wrote:", audio_16k)



# 5. Load Models

## 5.1 Load Whisper Model for Automatic Speech Recognition (ASR)

Load the **Whisper model for ASR** with the given device ("cpu" or "cuda") and precision type ("float16", "int8", etc.). You can select any [Whisper model](https://github.com/openai/whisper) size (e.g., "tiny" to "large-v3"; see section "Available models and languages" in the GitHub Repository) or provide a custom/fine-tuned model.

In [ ]:
model = WhisperModel("large-v3",  # "tiny", ...
                     device, 
                     compute_type=compute_type)

With faster-whisper, you can run transcription "normally" or with batched inference. Batched inference is mainly a speed option for GPUs (it processes several audio chunks at once). On CPU, it usually provides little benefit, so the default (non-batched) mode is typically used. If you want the batched model (alternative):

In [ ]:
#model = BatchedInferencePipeline(model=model)

## 5.2 Load pyannote Model for Diarization

Load the **pyannote speaker diarization pipeline from Hugging Face** using your access token. This model will later predict a timeline of "who speaks when" in the audio.

In [ ]:
diarization = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=own_token
)

# 6. Run Transcription / ASR

The `transcribe()` function takes an audio input and produces a **transcription as a sequence of time-stamped text segments.**

You can optionally provide a **language code**. It is one of **many optional settings**. Most of the other parameters control how the decoding is done. In the setup below:

- `word_timestamps`: If enabled, the output includes estimated start/end times per word (and other results, depending on the Whisper implementation used). This increases compute cost, and word timing can be less stable for very short filler sounds or noisy speech.
- `vad_filter`: If enabled, the system tries to skip non-speech regions, which often speeds up transcription and can reduce spurious text during silence/noise. Depending on sensitivity, it may also remove very short hesitation sounds (e.g., "um", "ähm").
- `beam_size`: Influences how many alternative decoding paths are explored. Larger values can sometimes improve accuracy, but they increase runtime and the benefit varies by audio.

For details on all available parameters and their defaults, refer to the [faster-whisper](https://github.com/SYSTRAN/faster-whisper) documentation.

>Example More Optional Settings: 
>
>This example shows how you can use "hotwords" to make the transcription model pay special attention to short hesitation sounds or filler phrases in your audio.
>
>In [noScribe](https://noscribe.de/de/), **hotwords** from a separate file are passed into `transcribe()` via the `hotwords` parameter to implement the "disfluencies" on/off toggle in the noScribe application. In faster-whisper, these hotwords are inserted as extra prompt tokens before decoding, which slightly biases the decoder toward producing those words when the audio is uncertain (for example, very short filler sounds like "um" or "ähm" that can be hard to distinguish from breathing or background noise). The original OpenAI [Whisper](https://github.com/openai/whisper) implementation does not provide a `hotwords` parameter under that name, so this behavior is specific to faster-whisper. The closest equivalent in the original Whisper implementation is providing a prompt via the `initial_prompt` parameter to bias decoding in a similar direction. 
>
>If you want to test how hesitation sounds can be encouraged in the `transcribe()` call below, add a `hotwords` parameter (German example: `hotwords="Äh, das ist, es ist, ähm, nicht so einfach."`; English example: `hotwords="Uhm, okay, here's what I'm, like, thinking."`, taken from noScribe’s [prompt.yml](https://github.com/kaixxx/noScribe/blob/main/prompt.yml)). Then compare the transcription results with vs. without hotwords on an audio file that contains such fillers.

> `!` Check that no or the correct language is selected.

In [ ]:
segments, info = model.transcribe(
    audio_16k,
    language="en",   # ADAPT language
    word_timestamps=False, 
    vad_filter=True, 
    beam_size=5
)

segments = list(segments)  # The transcription will actually run here

`print(segments)` shows a Python list of Segment objects. In faster-whisper **transcription results**, each text **segment** (or word) includes:
- start and end time, 
- recognized text,
- the underlying token IDs (see text box below for further information),
- and several scores that can be inspected if needed. These scores are model-internal confidence signals derived from the token probabilities during decoding.

In [ ]:
print(segments) # Show results

>Inspect Whisper Tokens:
>
>Whisper does not produce text directly. Internally, it predicts a sequence of tokens. These are numbered IDs that refer to entries in Whisper's fixed vocabulary (often whole words, parts of words, spaces, or punctuation). The final transcript is created by decoding these token IDs back into readable text.
>
>If you want to inspect the tokens in the model output more closely, you can look up what the token numbers correspond to by decoding them with the Whisper tokenizer. This lets you see the exact token sequence behind a segment. To do so, comment out the code below and insert a few tokens inside the `ids` list.

In [ ]:
#import whisper
#from whisper.tokenizer import get_tokenizer
#
#tokenizer = get_tokenizer(multilingual=True, language="en")
#
#ids = [51090, 1042, 11]                      # example: INSERT TOKEN(S)
#print([tokenizer.decode([i]) for i in ids])  # print token-by-token (roughly)

# 7. Run Speaker Diarization

Next, we run the **diarization model** on the audio file. This gives us a **timeline of who speaks when**, for example: *SPEAKER_00 speaks from 0–10 seconds, then SPEAKER_01 speaks from 10–15 seconds, and so on.* By setting `min_speakers` and `max_speakers`, we constrain the output to exactly that **number of speaker labels**, even if the real audio might contain fewer or more speakers.

In Sect. 4, `imageio_ffmpeg` provides an ffmpeg executable that we call directly to convert audio files. For diarization, `pyannote.audio`, loads audio files via its own internal decoder (see [pyannote.audio](https://github.com/pyannote/pyannote-audio)). This decoder can fail on some machines even when ffmpeg itself works. To make the notebook more robust (especially on Windows and some JupyterHub setups), we load the audio into memory with `torchaudio` (waveform & sample rate) and pass these values directly to the diarization instead of letting `pyannote.audio` open the file itself.

In [ ]:
def load_for_pyannote(path):
    """Load audio into memory (RAM) and output waveform shape: (channels, samples)."""
    waveform, sr = torchaudio.load(path)
    return {"waveform": waveform, "sample_rate": sr}

# Run speaker diarization on the audio:
diarization_result = diarization(
    load_for_pyannote(audio_16k),
    min_speakers=2,  # ADAPT minimum speaker number         
    max_speakers=2   # ADAPT maximum speaker number
)

`diarization_result.speaker_diarization` is a pyannote object. It lists who spoke when. Each line/entry is a time interval (start --> end) where the diarization model detected speech, and SPEAKER_x (e.g., SPEAKER_00) is the assigned speaker label.

In [ ]:
print(diarization_result.speaker_diarization) # Show results

# 8. Merge Results

We merge ASR output with diarization results so that **each spoken segment (and optionally each word) is linked to a speaker label** (e.g., SPEAKER_00). If word-level timestamps are available, we assign speakers per word; otherwise we assign one speaker per segment. 

The code in this section illustrates a **manual preparation** of transcription and diarization outputs and merging strategy and **can be adapted** if requirements differ.

## 8.1 ASR Result Formatting

Since the output of faster-whisper is stored in its own custom Python object, we first convert it into a **Python data structure (a list of dictionaries)**. We define a function `to_whisper_result` that extracts only the fields we need. This makes the transcript easier to inspect, save, and process in later steps. Each segment gets a start time, an end time, the transcribed text, and (if word-level timestamps were enabled) a list of words with their own timing information.

In [ ]:
def to_whisper_result(segments):
    """Convert faster-whisper segment objects into a simple list of dicts."""
    out = []
    for s in segments:
        item = {"start": float(s.start), "end": float(s.end), "text": s.text or ""}
        if getattr(s, "words", None):
            item["words"] = [
                {"word": w.word, "start": float(w.start), "end": float(w.end)}
                for w in s.words
                if w.start is not None and w.end is not None
            ]
        out.append(item)
    return out

Run conversion:

In [ ]:
asr_result = to_whisper_result(segments)

In [ ]:
print(asr_result) # Show results

## 8.2 Diarization Result Formatting

The pyannote diarization output is also stored in a custom object format. Before merging, we convert it into a simple **list of speaker time intervals**. Each entry is a **tuple** containing:
- start time
- end time
- speaker label

This makes the overlap comparison with the transcript easier and avoids repeatedly iterating over the pyannote object.

In [ ]:
turns = []
ann = diarization_result.speaker_diarization

for turn, _, spk in ann.itertracks(yield_label=True):
    turns.append((float(turn.start), float(turn.end), spk))

In [ ]:
print(turns) # Show results

## 8.3 Align Results

We **combine the converted ASR output with the diarization output** in the `align` function to create a speaker-attributed transcript. The result is a **list of dictionaries**, one per ASR segment, each containing:
- start,
- end,
- text,
- the assigned speaker,
- and an overlap flag.

If word-level timestamps are available, the same alignment is also applied at the word level (assigning speakers per word and deriving the segment’s main speaker from the words). We assign the speaker for each segment based on total overlapping duration (summed over words), or, if no words are available, based on maximum overlap with the segment interval. We mark overlap=True when two or more different speakers overlap with the interval at any point.

The alignment shown here is an **example** and can be adapted to different needs. For example, you could flatten the "words" lists here and continue processing and saving the results per word (instead of per segment) in Sect. 9, or merge adjacent segments with the same speaker into longer turns.

>Important: Diarization segments and ASR segments are independent time partitions, so they rarely match perfectly. The merge therefore compares time intervals and links them based on temporal overlap.

In [ ]:
def align(asr_result, turns):
    """
    Merge ASR output with diarization turns.

    asr_result: list of ASR segments (each dict has start/end/text, optional words)
    turns: list of diarization intervals (turn_start, turn_end, speaker)

    We work with time intervals:
    - ASR segments/words: [start (s), end (e)]
    - diarization speaker turns: [turn_start (ts), turn_end (te)]
    """

    def overlap_seconds(s, e, ts, te):
        """Return overlap duration (in seconds) between [s, e] and [ts, te]."""
        return max(0.0, min(e, te) - max(s, ts))

    def best_speaker(s, e):
        """Return the speaker with the largest total overlap with [s, e]."""
        overlap_by_speaker = {}
        for ts, te, spk in turns:
            ov = overlap_seconds(s, e, ts, te)
            if ov > 0:
                overlap_by_speaker[spk] = overlap_by_speaker.get(spk, 0.0) + ov
        return max(overlap_by_speaker, key=overlap_by_speaker.get) if overlap_by_speaker else None

    def is_overlapped(s, e):
        """True if >= 2 different speakers overlap with [s, e]."""
        seen = set()
        for ts, te, spk in turns:
            if overlap_seconds(s, e, ts, te) > 0:
                seen.add(spk)
                if len(seen) >= 2:
                    return True
        return False

    # Output: list of enriched ASR segments
    out = []

    # Loop over each ASR segment (NOTE: asr_result is now a LIST, not a dict)
    for seg in asr_result:

        # 1) Basic info: start/end time and full text
        new_seg = {
            "start": seg["start"],
            "end": seg["end"],
            "text": seg["text"],
        }

        # 2) Segment-level overlap flag
        new_seg["overlap"] = is_overlapped(seg["start"], seg["end"])

        # 3) Speaker labels
        words = seg.get("words", [])
        if words:
            new_words = []
            dur_by_speaker = {}  # total word duration per speaker within this ASR segment

            for w in words:
                spk = best_speaker(w["start"], w["end"])

                wd = dict(w)
                wd["speaker"] = spk
                wd["overlap"] = is_overlapped(w["start"], w["end"])
                new_words.append(wd)

                if spk is not None:
                    dur_by_speaker[spk] = dur_by_speaker.get(spk, 0.0) + (w["end"] - w["start"])

            new_seg["words"] = new_words
            new_seg["speaker"] = max(dur_by_speaker, key=dur_by_speaker.get) if dur_by_speaker else None
        else:
            new_seg["speaker"] = best_speaker(seg["start"], seg["end"])

        out.append(new_seg)

    return out

Run aligner:

In [ ]:
final_result = align(asr_result, turns)

In [ ]:
print(final_result) # Show results

# 9. Save Transcript

Finally, we save a **readable transcript as a text file**. For each segment, we write the speaker label, the transcribed text, and the segment start time. If a segment overlaps with another speaker, we add a tag. This produces a *SPEAKER: text [OVERLAP] [time]* format.

You **can adapt the format** depending on what you need for analysis in Python or external tools.

In [ ]:
# Save as ...
output_folder = "../Results/"
txt_path = os.path.join(output_folder, f"{file_name}.txt")

# Save 
with open(txt_path, "w", encoding="utf-8") as f:
    for seg in final_result:
        speaker = seg.get("speaker")
        text = seg["text"].strip()
        start = str(timedelta(seconds=seg["start"]))[:-3]
        overlap_flag = " [OVERLAP]" if seg.get("overlap", False) else ""
        f.write(f"{speaker}: {text}{overlap_flag} [{start}]\n\n")

print("Saved transcript to", txt_path)

If you would only run Whisper transcription (without diarization), you can save the result directly like this:

In [ ]:
# Save as ...
#output_folder = "../Results/"
#os.makedirs(output_folder, exist_ok=True)
#txt_path = os.path.join(output_folder, f"{file_name}.txt")

# Save
#with open(txt_path, "w", encoding="utf-8") as f:
#    for s in segments:
#        start = str(timedelta(seconds=float(s.start)))[:-3]
#        f.write(f"[{start}] {s.text.strip()}\n")

#print("Saved transcript to", txt_path)